# Create the Corpus Problem Lists

The notebook below creates the lists of documents for the corpus and datatype. This is used for iterating across the documents in the corpus when creating jobscripts that pull the known and unknown documents. To run this for a new set of data simply change the **corpus** and **data_type** parameters.

In [127]:
import sys
import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from utils import get_base_location, build_metadata_df, apply_temp_doc_id
from read_and_write_docs import read_jsonl, read_rds

In [128]:
corpus      = "Koppel's Blogs"
data_type   = "test"

# Set NAS so can run on Windows laptop seamlessly
nas_base_loc = get_base_location()

# --- set your args (strings) ---
nas_base_loc = "/Users/user/Documents/uni_work_offline"

known_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}/known_raw.jsonl"
unknown_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}/unknown_raw.jsonl"
metadata_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/metadata.rds"

save_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}"

## Read Data

In [129]:
metadata = read_rds(metadata_loc)
filtered_metadata = metadata[metadata['corpus'] == corpus]

known = read_jsonl(known_loc)
unknown = read_jsonl(unknown_loc)

In [130]:
filtered_metadata

,problem,corpus,known_author,unknown_author
3020,Author_189 vs Author_189,Koppel's Blogs,Author_189,Author_189
3021,Author_189 vs Author_19,Koppel's Blogs,Author_189,Author_19
3022,Author_19 vs Author_19,Koppel's Blogs,Author_19,Author_19
3023,Author_19 vs Author_190,Koppel's Blogs,Author_19,Author_190
3024,Author_190 vs Author_190,Koppel's Blogs,Author_190,Author_190
...,...,...,...,...
4815,Author_997 vs Author_998,Koppel's Blogs,Author_997,Author_998
4816,Author_998 vs Author_998,Koppel's Blogs,Author_998,Author_998
4817,Author_998 vs Author_999,Koppel's Blogs,Author_998,Author_999
4818,Author_999 vs Author_189,Koppel's Blogs,Author_999,Author_189


## Create Agg Problem Level Metadata

In [131]:
agg_problems = filtered_metadata['problem'].drop_duplicates()

## Create Metadata

Quite a convoluted process.

In [132]:
# Build the dataframe
complete_metadata = build_metadata_df(filtered_metadata, known, unknown)

# Set blank text column for function to work
complete_metadata['text'] = ''

# Rename the known column and create the new doc_id
complete_metadata.rename(columns={"known_doc_id": "orig_doc_id"}, inplace=True)
complete_metadata = apply_temp_doc_id(complete_metadata)
complete_metadata.rename(columns={
    "orig_doc_id": "orig_known_doc_id",
    "doc_id": "known_doc_id",
    "unknown_doc_id": "orig_doc_id"
}, inplace=True)

# Do the same for the unknown
complete_metadata = apply_temp_doc_id(complete_metadata)
complete_metadata.rename(columns={
    "orig_doc_id": "orig_unknown_doc_id",
    "doc_id": "unknown_doc_id",
}, inplace=True)

# Sort columns
complete_metadata = complete_metadata[["sample_id", "problem", "corpus", "known_doc_id", "unknown_doc_id"]]

## View the data

In [133]:
complete_metadata.head()

,sample_id,problem,corpus,known_doc_id,unknown_doc_id
0,1,Author_189 vs Author_189,Koppel's Blogs,author_189_post_1,author_189_post_4
1,2,Author_189 vs Author_189,Koppel's Blogs,author_189_post_2,author_189_post_4
2,3,Author_189 vs Author_189,Koppel's Blogs,author_189_post_3,author_189_post_4
3,4,Author_189 vs Author_19,Koppel's Blogs,author_189_post_1,author_19_post_2
4,5,Author_189 vs Author_19,Koppel's Blogs,author_189_post_2,author_19_post_2


## Create the Document Lists

In [134]:
known_doc_list = pd.Series(complete_metadata["known_doc_id"].astype(str))
unknown_doc_list = pd.Series(complete_metadata["unknown_doc_id"].astype(str))
problem_doc_list = known_doc_list + ' vs ' + unknown_doc_list

## Get Number of Rows in the Dataset

This is used for the jobscript.

In [135]:
num_rows_for_jobscript = complete_metadata.shape[0]
print(f"Number of rows needed in jobscript: {num_rows_for_jobscript}")

Number of rows needed in jobscript: 5400


## Save the Lists

In [136]:
print(f"Writing agg problem list - {len(agg_problems)} Aggregated Problems")
agg_problems.to_csv(f"{save_loc}/agg_problem_list.txt", index=False, header=False)
print(f"Writing problem doc list - {len(problem_doc_list)} Problems")
problem_doc_list.to_csv(f"{save_loc}/problem_doc_list.txt", index=False, header=False)
print(f"Writing known doc list - {len(known_doc_list)} Documents")
known_doc_list.to_csv(f"{save_loc}/known_doc_list.txt", index=False, header=False)
print(f"Writing unknown doc list - {len(unknown_doc_list)} Documents")
unknown_doc_list.to_csv(f"{save_loc}/unknown_doc_list.txt", index=False, header=False)
print("Wrote doc lists")

Writing agg problem list - 1800 Aggregated Problems
Writing problem doc list - 5400 Problems
Writing known doc list - 5400 Documents
Writing unknown doc list - 5400 Documents
Wrote doc lists
